# TrackPulse — Bronze to Streaming Silver (Incremental Job)

**Purpose:** Process new Bronze Delta changes into Streaming Silver and Quarantine.

**Schedule:** Run only after `01_bronze_capture_to_delta_incremental.ipynb` succeeds.

**Checkpoint:** Reuse `checkpoints/databricks_streaming/silver_incremental_v1/` exactly as configured in the processing cell. Never delete or reset it.

**Important:** This is the existing tested processing cell extracted from the current incremental notebook. It does not target the 10-million-row Historical Silver path.

ADF executes all cells in this notebook from top to bottom. Do not add exploratory CDF setup/test cells before the processing cell.

In [0]:

# Purpose: Process available Bronze events incrementally into Silver and Quarantine.
# Safety: Never overwrite historical Silver, delete tables, or reset checkpoints.
# Idempotency: Use stable keys and update existing records only when values change.

from pyspark.sql import functions as F
from pyspark.sql.window import Window
from delta.tables import DeltaTable

# ------------------------------------------------------------
# 1. Define the existing storage paths
# ------------------------------------------------------------

bronze_path = (
    "abfss://trackpulse-data@trackpulsestorage.dfs.core.windows.net/"
    "bronze/telemetry_stream/"
)

silver_path = (
    "abfss://trackpulse-data@trackpulsestorage.dfs.core.windows.net/"
    "silver/telemetry_stream/"
)

quarantine_path = (
    "abfss://trackpulse-data@trackpulsestorage.dfs.core.windows.net/"
    "silver/quarantine/telemetry_stream/"
)

# Reuse the existing checkpoint. Do not delete or reset it.
silver_checkpoint_path = (
    "abfss://trackpulse-data@trackpulsestorage.dfs.core.windows.net/"
    "checkpoints/databricks_streaming/silver_incremental_v1/"
)

# ------------------------------------------------------------
# 2. Validate the existing Delta tables
# ------------------------------------------------------------

bronze_df = spark.read.format("delta").load(bronze_path)
silver_df = spark.read.format("delta").load(silver_path)
quarantine_df = spark.read.format("delta").load(quarantine_path)

required_bronze_columns = [
    "event_id",
    "event_timestamp",
    "vehicle_id",
    "latitude",
    "longitude",
    "speed_kph",
    "odometer_km",
    "rpm",
    "fuel_rate_lph",
    "engine_temp_c",
    "ignition_on",
    "trip_id",
    "shipment_id",
    "driver_id",
    "route_id",
    "cargo_temp_c",
    "event_timestamp_utc",
    "bronze_ingested_at",
    "source_format",
    "event_json",
    "capture_sequence_number",
    "capture_enqueued_time_utc",
]

missing_columns = [
    c for c in required_bronze_columns
    if c not in bronze_df.columns
]

if missing_columns:
    raise ValueError(
        f"Bronze schema is missing required columns: {missing_columns}"
    )

expected_silver_columns = [
    "event_id",
    "event_timestamp",
    "vehicle_id",
    "latitude",
    "longitude",
    "speed_kph",
    "odometer_km",
    "rpm",
    "fuel_rate_lph",
    "engine_temp_c",
    "ignition_on",
    "trip_id",
    "shipment_id",
    "driver_id",
    "route_id",
    "cargo_temp_c",
    "ingested_at",
    "source_file",
]

if set(silver_df.columns) != set(expected_silver_columns):
    raise ValueError(
        "Silver schema differs from the expected schema. "
        "Inspect it before running writes."
    )

print("Bronze schema validated.")
print("Silver schema validated.")
print("Existing Bronze records:", bronze_df.count())
print("Existing Silver records:", silver_df.count())
print("Existing Quarantine records:", quarantine_df.count())

# ------------------------------------------------------------
# 3. Add missing Quarantine metadata columns if necessary
# ------------------------------------------------------------

if "quarantine_key" not in quarantine_df.columns:
    spark.sql(f"""
        ALTER TABLE delta.`{quarantine_path}`
        ADD COLUMNS (quarantine_key STRING)
    """)

if "dq_reasons" not in quarantine_df.columns:
    spark.sql(f"""
        ALTER TABLE delta.`{quarantine_path}`
        ADD COLUMNS (dq_reasons STRING)
    """)

# ------------------------------------------------------------
# 4. Backfill only missing Quarantine keys
# ------------------------------------------------------------

# Keep existing keys unchanged.
# Use the same fallback payload fields used by the streaming function.

spark.sql(f"""
    UPDATE delta.`{quarantine_path}`
    SET quarantine_key =
        CASE
            WHEN event_id IS NOT NULL AND trim(event_id) != ''
                THEN concat('event:', event_id)
            ELSE concat(
                'payload:',
                sha2(
                    coalesce(
                        event_json,
                        to_json(named_struct(
                            'capture_sequence_number',
                                capture_sequence_number,
                            'capture_enqueued_time_utc',
                                capture_enqueued_time_utc,
                            'event_id', event_id,
                            'event_timestamp', event_timestamp,
                            'vehicle_id', vehicle_id,
                            'latitude', latitude,
                            'longitude', longitude,
                            'speed_kph', speed_kph,
                            'odometer_km', odometer_km,
                            'rpm', rpm,
                            'fuel_rate_lph', fuel_rate_lph,
                            'engine_temp_c', engine_temp_c,
                            'ignition_on', ignition_on,
                            'trip_id', trip_id,
                            'shipment_id', shipment_id,
                            'driver_id', driver_id,
                            'route_id', route_id,
                            'cargo_temp_c', cargo_temp_c
                        ))
                    ),
                    256
                )
            )
        END
    WHERE quarantine_key IS NULL OR trim(quarantine_key) = ''
""")

# ------------------------------------------------------------
# 5. Define the incremental batch-processing function
# ------------------------------------------------------------

def process_silver_batch(batch_df, batch_id):

    # Skip empty batches.
    if batch_df.limit(1).count() == 0:
        print(f"Batch {batch_id}: empty batch; nothing to process.")
        return

    # Parse timestamps and retain ingestion metadata.
    prepared_df = (
        batch_df
        .withColumn(
            "event_timestamp_parsed",
            F.coalesce(
                F.to_timestamp(F.col("event_timestamp")),
                F.col("event_timestamp_utc")
            )
        )
        .withColumn(
            "ingested_at",
            F.coalesce(
                F.col("bronze_ingested_at"),
                F.current_timestamp()
            )
        )
    )

    # Evaluate data-quality rules.
    prepared_df = prepared_df.withColumn(
        "dq_reasons",
        F.concat_ws(
            "; ",
            F.when(
                F.col("event_id").isNull()
                | (F.trim(F.col("event_id")) == ""),
                F.lit("missing_event_id")
            ),
            F.when(
                F.col("event_timestamp_parsed").isNull(),
                F.lit("invalid_event_timestamp")
            ),
            F.when(
                F.col("latitude").isNull()
                | (F.col("latitude") < -90)
                | (F.col("latitude") > 90),
                F.lit("invalid_latitude")
            ),
            F.when(
                F.col("longitude").isNull()
                | (F.col("longitude") < -180)
                | (F.col("longitude") > 180),
                F.lit("invalid_longitude")
            ),
            F.when(
                F.col("speed_kph") < 0,
                F.lit("negative_speed")
            ),
            F.when(
                F.col("odometer_km") < 0,
                F.lit("negative_odometer")
            ),
            F.when(
                F.col("rpm") < 0,
                F.lit("negative_rpm")
            ),
            F.when(
                F.col("fuel_rate_lph") < 0,
                F.lit("negative_fuel_rate")
            )
        )
    )

    # Keep valid records only for Silver.
    valid_df = prepared_df.filter(F.col("dq_reasons") == "")

    # Keep one deterministic source row per event_id in this batch.
    valid_window = Window.partitionBy("event_id").orderBy(
        F.col("bronze_ingested_at").desc_nulls_last(),
        F.col("capture_sequence_number").desc_nulls_last()
    )

    valid_df = (
        valid_df
        .withColumn("_row_number", F.row_number().over(valid_window))
        .filter(F.col("_row_number") == 1)
        .select(
            "event_id",
            F.col("event_timestamp_parsed").alias("event_timestamp"),
            "vehicle_id",
            "latitude",
            "longitude",
            "speed_kph",
            "odometer_km",
            "rpm",
            "fuel_rate_lph",
            "engine_temp_c",
            "ignition_on",
            "trip_id",
            "shipment_id",
            "driver_id",
            "route_id",
            "cargo_temp_c",
            "ingested_at",
            F.lit(None).cast("string").alias("source_file")
        )
    )

    # Update Silver only when an existing event has changed.
    silver_compare_columns = [
        "event_timestamp",
        "vehicle_id",
        "latitude",
        "longitude",
        "speed_kph",
        "odometer_km",
        "rpm",
        "fuel_rate_lph",
        "engine_temp_c",
        "ignition_on",
        "trip_id",
        "shipment_id",
        "driver_id",
        "route_id",
        "cargo_temp_c",
        "ingested_at",
        "source_file",
    ]

    silver_changed_condition = "NOT (" + " AND ".join(
        f"t.`{c}` <=> s.`{c}`"
        for c in silver_compare_columns
    ) + ")"

    if valid_df.limit(1).count() > 0:
        (
            DeltaTable.forPath(spark, silver_path)
            .alias("t")
            .merge(
                valid_df.alias("s"),
                "t.event_id = s.event_id"
            )
            .whenMatchedUpdateAll(
                condition=silver_changed_condition
            )
            .whenNotMatchedInsertAll()
            .execute()
        )

    # --------------------------------------------------------
    # Prepare invalid records for Quarantine
    # --------------------------------------------------------

    # Use a deterministic key for records without event IDs.
    key_payload_columns = [
        "capture_sequence_number",
        "capture_enqueued_time_utc",
        "event_id",
        "event_timestamp",
        "vehicle_id",
        "latitude",
        "longitude",
        "speed_kph",
        "odometer_km",
        "rpm",
        "fuel_rate_lph",
        "engine_temp_c",
        "ignition_on",
        "trip_id",
        "shipment_id",
        "driver_id",
        "route_id",
        "cargo_temp_c",
    ]

    payload_json = F.to_json(
        F.struct(*[F.col(c) for c in key_payload_columns])
    )

    quarantine_df_batch = (
        prepared_df
        .filter(F.col("dq_reasons") != "")
        .withColumn(
            "quarantine_key",
            F.when(
                F.col("event_id").isNotNull()
                & (F.trim(F.col("event_id")) != ""),
                F.concat(F.lit("event:"), F.col("event_id"))
            ).otherwise(
                F.concat(
                    F.lit("payload:"),
                    F.sha2(
                        F.coalesce(F.col("event_json"), payload_json),
                        256
                    )
                )
            )
        )
    )

    # Keep one row per quarantine key within this batch.
    quarantine_window = Window.partitionBy("quarantine_key").orderBy(
        F.col("bronze_ingested_at").desc_nulls_last(),
        F.col("capture_sequence_number").desc_nulls_last()
    )

    quarantine_df_batch = (
        quarantine_df_batch
        .withColumn("_row_number", F.row_number().over(quarantine_window))
        .filter(F.col("_row_number") == 1)
        .drop("_row_number", "event_timestamp_parsed", "ingested_at")
    )

    # Align the source columns with the actual Quarantine schema.
    quarantine_target = spark.read.format("delta").load(quarantine_path)
    target_columns = quarantine_target.columns

    for column_name in target_columns:
        if column_name not in quarantine_df_batch.columns:
            quarantine_df_batch = quarantine_df_batch.withColumn(
                column_name,
                F.lit(None).cast(
                    quarantine_target.schema[column_name].dataType
                )
            )

    quarantine_df_batch = quarantine_df_batch.select(*target_columns)

    # Update Quarantine only when a stored value has changed.
    quarantine_compare_columns = [
        c for c in target_columns
        if c != "quarantine_key"
    ]

    quarantine_changed_condition = "NOT (" + " AND ".join(
        f"t.`{c}` <=> s.`{c}`"
        for c in quarantine_compare_columns
    ) + ")"

    if quarantine_df_batch.limit(1).count() > 0:
        (
            DeltaTable.forPath(spark, quarantine_path)
            .alias("t")
            .merge(
                quarantine_df_batch.alias("s"),
                "t.quarantine_key = s.quarantine_key"
            )
            .whenMatchedUpdateAll(
                condition=quarantine_changed_condition
            )
            .whenNotMatchedInsertAll()
            .execute()
        )

    print(f"Batch {batch_id} processed successfully.")

# ------------------------------------------------------------
# 6. Process available Bronze changes using the existing checkpoint
# ------------------------------------------------------------

if len(spark.streams.active) > 0:
    raise RuntimeError(
        "An active streaming query exists. Stop and inspect it "
        "before starting another query."
    )

bronze_stream = (
    spark.readStream
    .format("delta")
    .load(bronze_path)
)

silver_query = (
    bronze_stream.writeStream
    .foreachBatch(process_silver_batch)
    .option("checkpointLocation", silver_checkpoint_path)
    .trigger(availableNow=True)
    .start()
)

silver_query.awaitTermination()

print("Incremental Bronze-to-Silver processing finished.")


Bronze schema validated.
Silver schema validated.
Existing Bronze records: 167522
Existing Silver records: 166010
Existing Quarantine records: 12
Incremental Bronze-to-Silver processing finished.


In [0]:

# Purpose: Validate Silver and Quarantine after incremental processing.
# Safety: Read-only counts and duplicate checks.

silver_check = spark.read.format("delta").load(silver_path)
quarantine_check = spark.read.format("delta").load(quarantine_path)

print("Silver rows:", silver_check.count())
print("Silver distinct event IDs:", 
      silver_check.select("event_id").distinct().count())

print("Quarantine rows:", quarantine_check.count())
print("Quarantine distinct keys:",
      quarantine_check.select("quarantine_key").distinct().count())

print("Null Silver event IDs:",
      silver_check.filter(F.col("event_id").isNull()).count())

print("Null Quarantine keys:",
      quarantine_check.filter(F.col("quarantine_key").isNull()).count())

display(
    quarantine_check
    .groupBy("dq_reasons")
    .count()
    .orderBy(F.desc("count"))
)


Silver rows: 166010
Silver distinct event IDs: 166010
Quarantine rows: 12
Quarantine distinct keys: 12
Null Silver event IDs: 0
Null Quarantine keys: 0


dq_reasons,count
invalid_event_timestamp; invalid_latitude; invalid_longitude,12


In [0]:
# Inspect recent Delta Lake operations to verify incremental behavior

silver_path = (
    "abfss://trackpulse-data@trackpulsestorage.dfs.core.windows.net/"
    "silver/telemetry_stream/"
)

display(
    spark.sql(f"""
        DESCRIBE HISTORY delta.`{silver_path}`
    """).select(
        "version",
        "timestamp",
        "operation",
        "operationMetrics"
    ).limit(10)
)

version,timestamp,operation,operationMetrics
3,2026-10-28T19:38:36.000Z,MERGE,"Map(numTargetRowsCopied -> 0, numTargetRowsDeleted -> 0, numTargetFilesAdded -> 7, numTargetBytesAdded -> 5879986, numTargetBytesRemoved -> 0, numTargetDeletionVectorsAdded -> 2, numTargetRowsMatchedUpdated -> 1500, executionTimeMs -> 3589, materializeSourceTimeMs -> 870, numTargetRowsInserted -> 124500, numTargetRowsMatchedDeleted -> 0, numTargetDeletionVectorsUpdated -> 0, scanTimeMs -> 1019, numTargetRowsUpdated -> 1500, numOutputRows -> 126000, numTargetDeletionVectorsRemoved -> 0, numTargetRowsNotMatchedBySourceUpdated -> 0, numTargetChangeFilesAdded -> 0, numSourceRows -> 126000, numTargetFilesRemoved -> 0, numTargetRowsNotMatchedBySourceDeleted -> 0, rewriteTimeMs -> 1672)"
2,2026-10-28T18:08:01.000Z,MERGE,"Map(numTargetRowsCopied -> 0, numTargetRowsDeleted -> 0, numTargetFilesAdded -> 2, numTargetBytesAdded -> 1849532, numTargetBytesRemoved -> 1894236, numTargetDeletionVectorsAdded -> 0, numTargetRowsMatchedUpdated -> 41510, executionTimeMs -> 3736, materializeSourceTimeMs -> 924, numTargetRowsInserted -> 0, numTargetRowsMatchedDeleted -> 0, numTargetDeletionVectorsUpdated -> 0, scanTimeMs -> 1276, numTargetRowsUpdated -> 41510, numOutputRows -> 41510, numTargetDeletionVectorsRemoved -> 0, numTargetRowsNotMatchedBySourceUpdated -> 0, numTargetChangeFilesAdded -> 0, numSourceRows -> 41510, numTargetFilesRemoved -> 1, numTargetRowsNotMatchedBySourceDeleted -> 0, rewriteTimeMs -> 1490)"
1,2026-10-28T18:03:03.000Z,MERGE,"Map(numTargetRowsCopied -> 0, numTargetRowsDeleted -> 0, numTargetFilesAdded -> 0, numTargetBytesAdded -> 0, numTargetBytesRemoved -> 0, numTargetDeletionVectorsAdded -> 0, numTargetRowsMatchedUpdated -> 0, executionTimeMs -> 1831, materializeSourceTimeMs -> 17, numTargetRowsInserted -> 0, numTargetRowsMatchedDeleted -> 0, numTargetDeletionVectorsUpdated -> 0, scanTimeMs -> 0, numTargetRowsUpdated -> 0, numOutputRows -> 0, numTargetDeletionVectorsRemoved -> 0, numTargetRowsNotMatchedBySourceUpdated -> 0, numTargetChangeFilesAdded -> 0, numSourceRows -> 41510, numTargetFilesRemoved -> 0, numTargetRowsNotMatchedBySourceDeleted -> 0, rewriteTimeMs -> 1753)"
0,2026-10-28T14:30:11.000Z,WRITE,"Map(numFiles -> 1, numOutputRows -> 41510, numOutputBytes -> 1894236)"
